In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder


# LOAD DATA
data = pd.read_csv("agaricus-lepiota.data", header=None)

# First column = label (e=edible, p=poisonous), rest = features
y = data.iloc[:, 0]
X = data.iloc[:, 1:]

# TRAIN/TEST SPLIT — 75% train, 25% test, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size:  {len(X_test)}")

# PART 4.1 — TRAIN CUSTOM NAIVE BAYES

# Prior Probabilities P(Y=c)
classes = y_train.unique()

priors = {}
for c in classes:
    priors[c] = np.mean(y_train == c)

print("\n── Prior Probabilities ──")
for c, p in priors.items():
    label = "Edible" if c == "e" else "Poisonous"
    print(f"  P(Y={label}) = {p:.4f}")

# Conditional Probabilities P(X_i = x | Y = c) with Laplace Smoothing
# Formula: (count(X_i=x, Y=c) + 1) / (count(Y=c) + k)
# where k = number of unique values for feature X_i

cond_probs = {}

for col in X_train.columns:
    cond_probs[col] = {}
    values = X_train[col].unique()
    k = len(values)  # number of unique values for Laplace denominator

    for c in classes:
        cond_probs[col][c] = {}
        subset = X_train[y_train == c]
        total = len(subset)

        for v in values:
            count = np.sum(subset[col] == v)
            # Laplace smoothing
            cond_probs[col][c][v] = (count + 1) / (total + k)

# Print example: show P(X_1 = x | Y) for first feature
example_col = list(cond_probs.keys())[0]

rows = []
for c in classes:
    label = "Edible" if c == "e" else "Poisonous"
    for v, prob in cond_probs[example_col][c].items():
        rows.append({
            "Feature Value": v,
            "Class": label,
            "Probability": round(prob, 4)
        })

df_display = pd.DataFrame(rows)

print(f"\nConditional probabilities for feature {example_col}:")
print(df_display)

Training set size: 6093
Testing set size:  2031

── Prior Probabilities ──
  P(Y=Poisonous) = 0.4820
  P(Y=Edible) = 0.5180

Conditional probabilities for feature 1:
   Feature Value      Class  Probability
0              x  Poisonous       0.4410
1              k  Poisonous       0.1485
2              f  Poisonous       0.3955
3              b  Poisonous       0.0129
4              s  Poisonous       0.0003
5              c  Poisonous       0.0017
6              x     Edible       0.4684
7              k     Edible       0.0547
8              f     Edible       0.3751
9              b     Edible       0.0936
10             s     Edible       0.0079
11             c     Edible       0.0003


In [28]:

# PART 4.2 — PREDICT ON TEST SET
# Estimate P(Y=c | X) for each test point using stored
# prior and conditional probabilities. Use log probabilities
# to prevent numerical underflow.

def predict_row(row):
    """Return the class with the highest log-posterior probability."""
    log_posteriors = {}
    for c in classes:
        # Start with log prior
        log_prob = np.log(priors[c])
        for col in X_train.columns:
            v = row[col]
            # Fallback for unseen values: use Laplace-style smoothing
            fallback = 1 / (len(cond_probs[col][c]) + 1)
            log_prob += np.log(cond_probs[col][c].get(v, fallback))
        log_posteriors[c] = log_prob
    return max(log_posteriors, key=log_posteriors.get)

y_pred_custom = X_test.apply(predict_row, axis=1)

print("\n── Sample Predictions (first 5) ──")
print(pd.DataFrame({"Actual": y_test.values[:5], "Predicted": y_pred_custom.values[:5]}))



── Sample Predictions (first 5) ──
  Actual Predicted
0      e         e
1      p         p
2      e         e
3      p         e
4      p         p


In [23]:
# PART 4.3 — METRICS FOR CUSTOM NAIVE BAYES
# Binary conversion: poisonous=1, edible=0
y_test_bin   = (y_test == "p").astype(int)
y_pred_bin   = (y_pred_custom == "p").astype(int)

acc_custom   = accuracy_score(y_test_bin, y_pred_bin)
prec_custom  = precision_score(y_test_bin, y_pred_bin)
rec_custom   = recall_score(y_test_bin, y_pred_bin)
f1_custom    = f1_score(y_test_bin, y_pred_bin)

print("\n 4.3 Custom Naive Bayes Performance ")
print(f"  Accuracy:  {acc_custom:.4f}")
print(f"  Precision: {prec_custom:.4f}")
print(f"  Recall:    {rec_custom:.4f}")
print(f"  F1 Score:  {f1_custom:.4f}")


 4.3 Custom Naive Bayes Performance 
  Accuracy:  0.9527
  Precision: 0.9911
  Recall:    0.9101
  F1 Score:  0.9489


In [24]:
# PART 4.4 — COMPARE WITH SKLEARN CategoricalNB
# CategoricalNB is the correct sklearn model for categorical
# features. alpha=1.0 applies Laplace smoothing, matching
# our custom implementation exactly.

# OrdinalEncoder converts string categories to integers
# required by CategoricalNB
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_enc = encoder.fit_transform(X_train)
X_test_enc  = encoder.transform(X_test)

nb_pkg = CategoricalNB(alpha=1.0)  # alpha=1 = Laplace smoothing
nb_pkg.fit(X_train_enc, y_train)
y_pred_pkg = nb_pkg.predict(X_test_enc)

y_pred_pkg_bin = (y_pred_pkg == "p").astype(int)

acc_pkg   = accuracy_score(y_test_bin, y_pred_pkg_bin)
prec_pkg  = precision_score(y_test_bin, y_pred_pkg_bin)
rec_pkg   = recall_score(y_test_bin, y_pred_pkg_bin)
f1_pkg    = f1_score(y_test_bin, y_pred_pkg_bin)

print("\n── 4.4 CategoricalNB (sklearn) Performance ──")
print(f"  Accuracy:  {acc_pkg:.4f}")
print(f"  Precision: {prec_pkg:.4f}")
print(f"  Recall:    {rec_pkg:.4f}")
print(f"  F1 Score:  {f1_pkg:.4f}")

# --- Side-by-side comparison table ---
comparison = pd.DataFrame({
    "Metric":      ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Custom NB":   [acc_custom, prec_custom, rec_custom, f1_custom],
    "sklearn NB":  [acc_pkg,    prec_pkg,    rec_pkg,    f1_pkg],
    "Difference":  [abs(acc_custom - acc_pkg),
                    abs(prec_custom - prec_pkg),
                    abs(rec_custom - rec_pkg),
                    abs(f1_custom - f1_pkg)]
})
print("\n── Comparison Table ──")
print(comparison.round(4).to_string(index=False))


── 4.4 CategoricalNB (sklearn) Performance ──
  Accuracy:  0.9527
  Precision: 0.9911
  Recall:    0.9101
  F1 Score:  0.9489

── Comparison Table ──
   Metric  Custom NB  sklearn NB  Difference
 Accuracy     0.9527      0.9527         0.0
Precision     0.9911      0.9911         0.0
   Recall     0.9101      0.9101         0.0
 F1 Score     0.9489      0.9489         0.0
